# 01 — Data exploration

**Goal:** understand the deadlock-api `match_player` data well enough to decide, for Milestone 2, which matches to keep, which rows to drop, and which features to build.

Exploratory data analysis (EDA) comes before modeling because a model can only be as good as our understanding of its inputs. In particular we're hunting for:
- **validity problems**: missing players, duplicates, abandons, remakes;
- **base rates**: how often each team wins when we know nothing;
- **leakage**: anything in a row that secretly reveals the answer;
- **signal**: does the game state actually predict the winner, and how does that change over time?

**Before running:** select the `Python (deadlock)` kernel, and create the local sample once with
`python -m deadlock_predictor.data.sample` (5,000 ranked matches, ~11 MB).

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from deadlock_predictor.data.dumps import USER_AGENT, connect, fetch_manifest, match_player_urls

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE = ROOT / "data" / "interim" / "sample_ranked.parquet"
assert SAMPLE.exists(), "Run `python -m deadlock_predictor.data.sample` first"

con = connect()  # DuckDB; sends the User-Agent the dump server requires
con.sql(f"CREATE OR REPLACE VIEW s AS SELECT * FROM '{SAMPLE.as_posix()}'")
plt.rcParams["figure.figsize"] = (9, 4)
pd.set_option("display.max_columns", 30)
# duckdb's .df() triggers a pandas 3.1 deprecation warning that isn't ours to fix
warnings.filterwarnings("ignore", message="The inplace keyword")

## 1. How the dumps are laid out

The manifest lists every Parquet file. `match_player` is partitioned by `match_id // 1,000,000`; match IDs grow over time, so **a higher partition means newer matches**. Each partition has one `base` file (rebuilt regularly, so file names change; always take URLs from a fresh manifest) and there are hourly `delta` files.

In [ ]:
manifest = fetch_manifest()
files = pd.DataFrame(manifest["tables"]["match_player"]["files"])
base = files[files.kind == "base"].sort_values("partition")
print(f"{len(base)} base files, {len(files) - len(base)} delta files")
print(
    f"{base.rows.sum():,} player rows ≈ {base.rows.sum() / 12:,.0f} matches, "
    f"{base.bytes.sum() / 1e9:,.0f} GB"
)
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(base.partition, base.rows / 12, width=1)
ax.set(title="Matches per partition", xlabel="partition (match_id // 1,000,000)", ylabel="matches");

## 2. Which matches are ranked 6v6?

One row per player per match, across every mode. We query the newest *complete* partition directly over HTTPS. DuckDB only downloads the columns it needs, so this takes seconds, not a 4 GB download.

In [ ]:
latest = int(base.loc[base.rows > 1_000_000, "partition"].max())
latest_urls = match_player_urls(
    manifest, min_partition=latest, max_partition=latest, include_deltas=False
)
modes = con.sql(f"""
    SELECT match_mode, game_mode, count(DISTINCT match_id) AS matches
    FROM read_parquet({latest_urls!r})
    GROUP BY ALL ORDER BY matches DESC
""").df()
modes["share"] = (modes.matches / modes.matches.sum()).round(3)
print(f"partition {latest}")
modes

Ranked 6v6 is `match_mode = 'Ranked' AND game_mode = 'Normal'`: roughly 15% of matches, but still about **16k ranked matches per day**.

The cell below (off by default, ~3 minutes) counts matches per day over the last ~20 partitions. It shows ranked volume flat while total volume roughly doubles around **late September**, which lines up with a major patch (section 10).

In [ ]:
RUN_SLOW_SCAN = False

if RUN_SLOW_SCAN:
    recent_urls = match_player_urls(manifest, min_partition=latest - 20, include_deltas=False)
    per_day = con.sql(f"""
        SELECT start_time::DATE AS day,
               count(DISTINCT match_id) FILTER (match_mode = 'Ranked' AND game_mode = 'Normal')
                   AS ranked,
               count(DISTINCT match_id) AS all_modes
        FROM read_parquet({recent_urls!r}) GROUP BY 1 ORDER BY 1
    """).df()
    per_day.set_index("day").plot(title="Matches per day", marker=".")

## 3. The local sample

5,000 ranked matches picked at random (by hashing `match_id`, so the same seed always gives the same sample) from three recent partitions, with ~45 of the 157 columns. Columns like `stats.net_worth` are **lists**: one value per snapshot in time.

In [ ]:
df = con.sql("SELECT * FROM s").df()
print(df.shape, "|", df.match_id.nunique(), "matches")
df[
    [
        "match_id",
        "start_time",
        "duration_s",
        "team",
        "hero_id",
        "kills",
        "deaths",
        "net_worth",
        "winning_team",
        "stats.time_stamp_s",
    ]
].head(12)

## 4. Integrity checks

A ranked match should have 12 players, 6 per team, each appearing once. Anything else is a data problem or an unusual match (leavers, abandons, remakes) that the model shouldn't learn from blindly.

In [ ]:
per_match = df.groupby("match_id").agg(
    players=("player_slot", "size"),
    team0_players=("team", lambda t: (t == "Team0").sum()),
    duration_s=("duration_s", "first"),
    abandons=("abandon_match_time_s", lambda a: (a > 0).sum()),
)
print("players per match:", per_match.players.value_counts().to_dict())
print("Team0 players per match:", per_match.team0_players.value_counts().to_dict())
print("duplicate (match_id, player_slot) rows:", df.duplicated(["match_id", "player_slot"]).sum())
print(f"matches with at least one abandon: {(per_match.abandons > 0).mean():.1%}")

In [ ]:
minutes = per_match.duration_s / 60
ax = minutes.plot.hist(bins=60)
ax.set(title="Match duration", xlabel="minutes")
minutes.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).round(1)

**Do base files already contain the delta rows?** If the newest delta file's matches are all in the matching base file, we can read base files only and skip deltas (simpler, no de-duplication needed).

In [ ]:
complete = set(base.loc[base.rows > 1_000_000, "partition"])
delta = files[files.kind == "delta"].sort_values("hi").iloc[-1]
counts = {int(p): n for p, n in delta.rows_by_partition.items() if int(p) in complete}
part = max(counts, key=counts.get)  # partition with a full base file and the most delta rows
delta_url = f"{manifest['public_url']}/{delta.key}"
base_url = match_player_urls(
    manifest, min_partition=part, max_partition=part, include_deltas=False
)[0]
print(f"newest delta file, partition {part} ({counts[part]:,} rows)")
con.sql(f"""
    SELECT count(*) AS delta_matches, count(b.match_id) AS also_in_base,
           min(d.created_at)::VARCHAR AS first_added, max(d.created_at)::VARCHAR AS last_added
    FROM (SELECT match_id, any_value(created_at) AS created_at FROM '{delta_url}'
          WHERE match_id // 1000000 = {part} GROUP BY 1) d
    LEFT JOIN (SELECT DISTINCT match_id FROM '{base_url}') b USING (match_id)
""").df()

**None of the delta file's matches are in the base file**, even for an old, complete partition. Deltas hold rows by *when they were added* (`created_at`, the last hour), not by when the match was played: old matches are still being backfilled. So the full table is base ∪ delta, and a given date range keeps gaining matches over time. The pipeline must read both and de-duplicate on `(match_id, player_slot)` in case a delta is later folded into a base file.

## 5. Labels and the base rate

The label is `winning_team`. If one side won much more often, "side" alone would be a feature. The 95% interval below tells us whether any gap is bigger than sampling noise.

The base rate also sets the bar for every model: always predicting 50% gives a log loss of ln 2 ≈ **0.693**. A model that can't beat that has learned nothing.

In [ ]:
matches = df.drop_duplicates("match_id").set_index("match_id")
p = (matches.winning_team == "Team0").mean()
n = len(matches)
print(f"Team0 win rate: {p:.3f} ± {1.96 * np.sqrt(p * (1 - p) / n):.3f} (95% CI, n={n})")
print(matches.match_outcome.value_counts().to_dict())

## 6. Snapshot timing and the leakage trap

Each player's `stats.*` lists are sampled at `stats.time_stamp_s`. These snapshots are the moments where we can know the game state, so they become our training examples.

In [ ]:
snaps = (
    df.drop_duplicates("match_id")[["match_id", "duration_s", "stats.time_stamp_s"]]
    .explode("stats.time_stamp_s")
    .rename(columns={"stats.time_stamp_s": "t"})
    .astype({"t": int})
)
snaps["is_last"] = snaps.t == snaps.groupby("match_id").t.transform("max")
last = snaps[snaps.is_last]
print("snapshots per match:", snaps.groupby("match_id").size().describe().round(1).to_dict())
print(f"last snapshot is exactly at match end: {(last.t == last.duration_s).mean():.1%}")
print("schedule (excluding the last snapshot):")
print(snaps[~snaps.is_last].t.value_counts().sort_index().head(16).to_dict())
same = df.groupby("match_id")["stats.time_stamp_s"].apply(lambda s: s.map(tuple).nunique() == 1)
print(f"all players in a match share the same timestamps: {same.mean():.1%}")

**The final snapshot is taken at the moment the match ends.** By then the winner has destroyed the enemy base and is usually far ahead, so the row effectively *contains the answer*. Training on it would make the model look much better than it is. This is **label leakage**, and we'll drop the end-of-match snapshot in the pipeline.

The regular snapshots are every 3 minutes until 15:00, then every 5 minutes, which gives our curve a point every 3–5 minutes.

## 7. Signal check: does a net worth lead predict the winner?

Net worth ("souls") is the most obvious summary of who's winning. We sum it per team at every snapshot (excluding the final one) and ask how often the team ahead goes on to win.

In [ ]:
team_nw = con.sql("""
    WITH long AS (
        SELECT match_id, team, winning_team, duration_s,
               unnest("stats.time_stamp_s") AS t, unnest("stats.net_worth") AS nw
        FROM s
    )
    SELECT match_id, t,
           any_value(duration_s) AS duration_s,
           any_value(winning_team) = 'Team0' AS team0_won,
           sum(nw) FILTER (team = 'Team0') - sum(nw) FILTER (team = 'Team1') AS nw_diff
    FROM long GROUP BY match_id, t
""").df()
team_nw = team_nw[team_nw.t < team_nw.duration_s]  # drop the end-of-match snapshot
team_nw["ahead_won"] = np.where(team_nw.nw_diff > 0, team_nw.team0_won, ~team_nw.team0_won)

by_t = team_nw[team_nw.nw_diff != 0].groupby("t").ahead_won.agg(["mean", "size"])
by_t = by_t[by_t["size"] >= 200]
ax = (by_t["mean"]).plot(marker="o")
ax.axhline(0.5, color="grey", ls="--")
ax.set(
    title="How often the team ahead in net worth wins",
    xlabel="game time (s)",
    ylabel="win rate",
    ylim=(0.45, 1),
);

In [ ]:
fig, ax = plt.subplots()
for t in [540, 900, 1500, 2100]:
    d = team_nw[team_nw.t == t]
    edge = d.nw_diff.abs().quantile(0.95)
    bins = pd.cut(d.nw_diff, np.linspace(-edge, edge, 13))
    rate = d.groupby(bins, observed=True).team0_won.agg(["mean", "size"])
    rate = rate[rate["size"] >= 30]
    ax.plot([b.mid for b in rate.index], rate["mean"], marker=".", label=f"{t // 60} min")
ax.axhline(0.5, color="grey", ls="--")
ax.set(
    title="P(Team0 wins) vs net worth lead",
    xlabel="Team0 − Team1 net worth",
    ylabel="Team0 win rate",
)
ax.legend();

Two things to notice:
1. **A lead predicts more as the game goes on**, but the same absolute lead means different things at 9 minutes and at 35 minutes. So the model needs game time as an input, and interactions between time and lead. That's the motivation for trying gradient boosting after the logistic baseline.
2. **The curves are S-shaped.** That's exactly the shape logistic regression models: P = σ(w · lead + b). The baseline should do reasonably well.
3. **The "team ahead wins" rate rises until ~35 minutes, then falls.** That's **selection bias**: matches still running at 45 minutes are mostly ones where the leader *failed* to close out, so long matches are close matches. Another reason the model needs game time as a feature.

## 8. Objectives and the mid-boss

Objective entries list the **team that owns the objective**, its type, and `destroyed_time_s` (0 = never destroyed). Every player row repeats the same team-level lists, so we read one row per match.

In [ ]:
ONE_ROW_PER_MATCH = """
    WITH one AS (
        SELECT * FROM s
        QUALIFY row_number() OVER (PARTITION BY match_id ORDER BY player_slot) = 1
    )
"""
obj = con.sql(
    ONE_ROW_PER_MATCH
    + """
    SELECT match_id, winning_team,
           unnest("objectives.team") AS owner,
           unnest("objectives.team_objective") AS objective,
           unnest("objectives.destroyed_time_s") AS destroyed_s
    FROM one
"""
).df()
obj["destroyed"] = obj.destroyed_s > 0
summary = obj.groupby("objective").agg(
    entries_per_match=("match_id", lambda m: len(m) / obj.match_id.nunique()),
    destroyed_rate=("destroyed", "mean"),
    median_destroyed_min=("destroyed_s", lambda x: x[x > 0].median() / 60),
)
summary.sort_values("median_destroyed_min").round(2)

In [ ]:
core = obj[(obj.objective == "Core") & obj.destroyed]
print(f"destroyed Core belongs to the losing team: {(core.owner != core.winning_team).mean():.1%}")
lost_first_t1 = (
    obj[obj.objective.str.startswith("Tier1") & obj.destroyed]
    .sort_values("destroyed_s")
    .groupby("match_id")
    .first()
)
rate = (lost_first_t1.owner != lost_first_t1.winning_team).mean()
print(f"team that loses the first Tier 1 tower goes on to lose: {rate:.1%}")

In [ ]:
mb = con.sql(
    ONE_ROW_PER_MATCH
    + """
    SELECT match_id, winning_team,
           unnest("mid_boss.team_killed") AS killed_by,
           unnest("mid_boss.team_claimed") AS claimed_by,
           unnest("mid_boss.destroyed_time_s") AS t
    FROM one
"""
).df()
print(
    "mid-boss kills per match:",
    mb.groupby("match_id").size().reindex(matches.index, fill_value=0).value_counts().to_dict(),
)
print(f"team that claims it wins: {(mb.claimed_by == mb.winning_team).mean():.1%}")
stolen = (mb.killed_by != mb.claimed_by).mean()
print(f"claimed by the team that didn't kill it (stolen): {stolen:.1%}")
print("kill time (min):", (mb.t / 60).describe().round(1).to_dict())

**Leakage again:** Core, Titan and shield generators fall at the very end of a match, so a feature like "objectives destroyed" must count only objectives with `0 < destroyed_s <= t`, never the whole-match totals (`objectives_mask_team*` are end-of-match masks, so they can't be features).

## 9. Heroes and rank

In [ ]:
heroes = (
    df.assign(won=df.team == df.winning_team)
    .groupby("hero_id")
    .agg(picks=("won", "size"), win_rate=("won", "mean"))
    .sort_values("picks", ascending=False)
)
heroes["pick_rate"] = heroes.picks / df.match_id.nunique()
heroes["ci95"] = 1.96 * np.sqrt(0.25 / heroes.picks)
print(len(heroes), "heroes")
ax = heroes.sort_values("win_rate").plot.barh(
    y="win_rate", xerr="ci95", legend=False, figsize=(7, 9)
)
ax.axvline(0.5, color="grey", ls="--")
ax.set(title="Hero win rate (95% CI)", xlabel="win rate");

Most hero win rates sit within noise of 50% at this sample size; the error bars are ±3–5 points. Hero identity will be a weak feature on its own, but **team composition** could matter more, and early in a match (before any gold is earned) it's the only information there is.

In [ ]:
df[["average_badge_team0", "average_badge_team1", "average_badge"]].describe().round(1)

The team-level `average_badge_team0/1` columns are **always 0** in recent data, but the per-player `average_badge` is populated (11–115). Badges look like *tier × 10 + subtier*; we'll confirm that in Milestone 2. We can compute each team's average rank ourselves and use the rank gap as a feature: it's the only skill information available before the match starts.

## 10. Patches: choosing a date range

Balance changes shift what wins, so training on old patches can teach outdated patterns. `game_mode_version` turns out to be constant, so we use the API's list of major patch days.

In [ ]:
print("game_mode_version values:", df.game_mode_version.unique())
big_days = requests.get(
    "https://api.deadlock-api.com/v1/patches/big-days",
    headers={"User-Agent": USER_AGENT},
    timeout=60,
).json()
big_days = pd.to_datetime(pd.Series(big_days)).sort_values(ascending=False).reset_index(drop=True)
print("latest major patches:", big_days.head(4).dt.strftime("%Y-%m-%d %H:%M UTC").tolist())
print("sample covers:", df.start_time.min(), "to", df.start_time.max())

## 11. Findings → Milestone 2 decisions

Numbers are from the 5,000-match sample (Oct 16–20, 2026) unless noted.

| Finding | Decision for the pipeline |
|---|---|
| Ranked 6v6 = `match_mode='Ranked'`, `game_mode='Normal'`, ~16k matches/day | Filter on both |
| Last major patch: **2026-09-29 20:25 UTC** (`game_mode_version` is useless) | Train on matches starting after it (~400k matches as of Oct 24) |
| Delta files hold matches not in base files | Read base **and** delta files; de-duplicate on `(match_id, player_slot)` |
| 0.5% of matches have 11 players; 5% have an abandon | Drop matches without exactly 6 + 6 players; keep a `has_abandon` flag and test both ways |
| Very short matches exist (min 7 min, 1% under 16 min) | Keep; they're legitimate stomps/surrenders, but check them in M2 |
| Team0 win rate 49.9% ± 1.4% | No side effect; the base rate is 50% (log loss 0.693 to beat) |
| Snapshots at 3, 6, 9, 12, 15 min, then every 5 min, all players aligned; ~9.6 per match | One training row per (match, snapshot) |
| **Last snapshot is always exactly at match end** | **Drop it** (label leakage) |
| Net worth lead is strongly predictive; its meaning depends on game time; S-shaped | Features: team net worth diff, game time; baseline = logistic regression |
| Objective lists give the owner team and destroy time (0 = never); Core always belongs to the loser | Objective features count only `0 < destroyed_s <= t`; never use `objectives_mask_*` |
| First Tier 1 lost → 60.9% loss; mid-boss claimer wins 76% | Features: objectives destroyed per team by type tier; mid-boss claims by t |
| 43 heroes, individual win rates mostly within noise | Hero/draft features come later (Milestone 4) |
| Per-player `average_badge` populated; team averages are 0 | Feature: team average badge difference |
